In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
path = "../data"
files = glob.glob(os.path.join(path, "*.csv"))

dfs = []

for file in files:
    df = pd.read_csv(file)
    
    ticker = os.path.basename(file).replace(".csv", "")
    df["Ticker"] = ticker
    
    dfs.append(df)

stocks = pd.concat(dfs, ignore_index=True)

In [ ]:
stocks["Date"] = pd.to_datetime(stocks["Date"])
stocks = stocks.sort_values(["Ticker", "Date"])
stocks = stocks.dropna()

In [ ]:
stocks = stocks[["Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]

In [ ]:
stocks["Return"] = stocks.groupby("Ticker")["Adj Close"].pct_change() * 100

In [ ]:
stocks["SMA_10"] = stocks.groupby("Ticker")["Adj Close"].transform(lambda x: x.rolling(10).mean())
stocks["SMA_20"] = stocks.groupby("Ticker")["Adj Close"].transform(lambda x: x.rolling(20).mean())
stocks["EMA_10"] = stocks.groupby("Ticker")["Adj Close"].transform(lambda x: x.ewm(span=10, adjust=False).mean())

In [ ]:
import numpy as np

def rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    
    rs = gain / loss
    return 100 - (100 / (1 + rs))

In [ ]:
stocks["RSI_14"] = stocks.groupby("Ticker")["Adj Close"].transform(rsi)

In [ ]:
def macd(series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    return ema12 - ema26

In [ ]:
stocks["MACD"] = stocks.groupby("Ticker")["Adj Close"].transform(macd)
stocks["Signal"] = stocks.groupby("Ticker")["MACD"].transform(lambda x: x.ewm(span=9, adjust=False).mean())

In [ ]:
df = stocks[stocks["Ticker"] == "AAPL"]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(df["Date"], df["Adj Close"], label="Price")
plt.plot(df["Date"], df["SMA_10"], label="SMA 10")
plt.plot(df["Date"], df["SMA_20"], label="SMA 20")

plt.legend()
plt.title("AAPL Price with Moving Averages")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["Date"], df["RSI_14"])
plt.axhline(70, color="red")
plt.axhline(30, color="green")
plt.title("AAPL RSI")
plt.show()